# Fire distribution across Australia

## Accessing Wildfire Data via API

In [2]:
# import necessary libraries
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt


In [3]:
# 1.
# access api url

## satellite: VIIRS SNPP NRT 
## area: 'world' = entire world 
## day range: '1' = data of one day
## date: None = most recent available data, so today's data

MAP_KEY = '4899a992545cbeb46f9fd0b6a025ef17'
area_url ='https://firms.modaps.eosdis.nasa.gov/api/area/csv/' + MAP_KEY + '/VIIRS_SNPP_NRT/world/1' # warum gehen 1, 3 oder 5 tage aber ab 8 oder so nicht mehr??

# 2.
# read in the data from URL

df_area = pd.read_csv(area_url)

# 3.
# have a first glimpse at the data

df_area.head(5)
df_area.shape

(13634, 14)

## Cleaning and Rearranging Data

### Filter for Data only within Australia

In [4]:
# define a bounding box that contains only the area of Australia based on its WGS84 coordinates

coords = [112, -44, 154, -9]

df_aus = df_area[(df_area['longitude'] >= coords[0]) & (df_area['latitude'] >= coords[1]) & (df_area['longitude'] <= coords[2]) & (df_area['latitude'] <= coords[3])].copy()
df_aus.shape
df_aus.head(20)
df_aus.tail()

,latitude,longitude,bright_ti4,scan,track,acq_date,acq_time,satellite,instrument,confidence,version,bright_ti5,frp,daynight
5286,-27.71891,114.16743,338.32,0.48,0.48,2026-05-04,644,N,VIIRS,n,2.0NRT,306.55,5.80,D
5287,-27.69001,114.18737,337.01,0.49,0.48,2026-05-04,644,N,VIIRS,n,2.0NRT,305.31,5.81,D
5288,-27.68900,114.19215,348.48,0.49,0.48,2026-05-04,644,N,VIIRS,n,2.0NRT,307.30,5.81,D
5289,-20.61557,116.77142,331.24,0.49,0.65,2026-05-04,646,N,VIIRS,n,2.0NRT,298.63,2.95,D
5290,-20.61496,116.77148,332.01,0.49,0.65,2026-05-04,646,N,VIIRS,n,2.0NRT,298.40,1.82,D


### Filter for required Timeframe

In [5]:
# 1. 
# combine the acq_date and acq_time column to one acq_datetime column and set it to an active time format with pandas function to_datetime

## acq_date is a string in the format YYYY-MM_DD, 
## while acq_time is an integer in Greenwich Mean Time (e.g. 603 meaning 6:03), 
## so it needs to be converted to string too (with astype(str)),
## fill it up to 4 numbers with zeros, so that all times have the same length (with str.zfill(4), e.g. 603 -> 0603)
## and save it as the format '%Y-%m-%d %H%M'

df_aus['acq_datetime'] = pd.to_datetime(df_aus['acq_date'] + ' ' + df_aus['acq_time'].astype(str).str.zfill(4), format='%Y-%m-%d %H%M')
df_aus.head()

print (f'Australia GMT timezone datetime value range: {df_aus['acq_datetime'].min()} to {df_aus['acq_datetime'].max()}')

# 2.
# convert GMT into local time?

# 3.
# # Set the timestamp column as the index ?
###hourly_data = hourly_data.set_index("timestamp")

# Notice how 'timestamp' drops down a level to become the index!
###display(hourly_data.head(3))


Australia GMT timezone datetime value range: 2026-05-04 03:17:00 to 2026-05-04 06:46:00


### Converting raw coordinates into geometries

In [6]:
# the projection EPSG:9473 is used for Australia, as it is recommended for national mapping

# convert latitude, longitude values into point geometry and set crs (since no crs extisting yet) with crs="EPSG:9473" to EPSG:9473

gdf_aus = gpd.GeoDataFrame(
    df_aus, geometry=gpd.points_from_xy(df_aus.longitude, df_aus.latitude), crs="EPSG:9473")
print(gdf_aus.crs)
gdf_aus.head()

EPSG:9473


,latitude,longitude,bright_ti4,scan,track,acq_date,acq_time,satellite,instrument,confidence,version,bright_ti5,frp,daynight,acq_datetime,geometry
997,-41.80918,147.28889,351.19,0.47,0.64,2026-05-04,317,N,VIIRS,n,2.0NRT,283.68,6.88,D,2026-05-04 03:17:00,POINT (147.289 -41.809)
998,-41.80903,147.29489,325.71,0.47,0.64,2026-05-04,317,N,VIIRS,n,2.0NRT,281.75,8.29,D,2026-05-04 03:17:00,POINT (147.295 -41.809)
1010,-37.29874,148.23022,339.28,0.39,0.59,2026-05-04,319,N,VIIRS,n,2.0NRT,289.82,4.24,D,2026-05-04 03:19:00,POINT (148.23 -37.299)
1011,-37.29628,148.23187,328.80,0.39,0.59,2026-05-04,319,N,VIIRS,n,2.0NRT,289.22,3.89,D,2026-05-04 03:19:00,POINT (148.232 -37.296)
1012,-34.30931,146.05737,330.10,0.49,0.65,2026-05-04,319,N,VIIRS,n,2.0NRT,291.11,2.00,D,2026-05-04 03:19:00,POINT (146.057 -34.309)


## Calculating means etc?

## Visualise it and create interactive Map

In [7]:
# try it with .explore
interactive_map = gdf_aus.explore(
    column="frp", # the column fire radiative power indicates the color
    cmap="viridis",
    alpha=0.8,
    tiles="CartoDB Positron",
    legend_kws={"Caption": "Radiative Power of Australian Wildfires"}
)
interactive_map

In [9]:
# alternative way: ?
##%pip install geodatasets cartopy
##from cartopy import crs as ccrs
##from geodatasets import get_path

##path = get_path("naturalearth.land")
##world = gpd.read_file(path)

##ax = world.plot(figsize=(10, 10), color="grey", edgecolor="black")
##ax.set_xlim([coords[0],  coords[2]])
##ax.set_ylim([coords[1],  coords[3]])